# Step 6: Full Simulation Runner

This notebook orchestrates the **complete end-to-end simulation**:

1. **Arrivals** — re-uses Sophia's queue logic (imported via `%run`).
   Patients are generated with demographics from `population.py`.
2. **Screening** (Steps 2–3) — eligibility, test assignment, results.
3. **Follow-up** (Steps 4–5) — routing, colposcopy, CIN grading, treatment.
4. **Exit / Re-entry** (Step 6) — treated → surveillance loop;
   untreated / LTFU → exit; unscreened + willing → rescheduled.

Metrics are written to `metrics_state` and passed to `05_metrics_outputs.ipynb`.

---
**Integration note**: Sophia's notebook is loaded with `%run`.
Our extended `Patient` class (from `patient.py`) is a superset of hers,
so all her queue functions work unchanged.

In [ ]:
import sys, random
from collections import defaultdict, deque
sys.path.insert(0, '../src')

import config as cfg
from patient import Patient
from population import sample_patient
from screening import (
    get_eligible_screenings,
    run_screening_step,
    handle_unscreened,
    is_due_for_screening,
)
from followup import run_cervical_followup, run_lung_followup, run_stub_followup
from metrics import initialize_metrics, record_screening, record_exit, print_summary

print('Core imports OK')

## Load Sophia's Arrival Functions

`%run` imports all of Sophia's functions and global vars into this namespace.
We then override patient creation to use our enriched `Patient` class.

In [ ]:
%run "../reference/initial_model_NYP_flow_simulation (1).ipynb"

# Confirm Sophia's key functions are loaded
print('Sophia functions loaded:', [f for f in dir() if f in (
    'initialize_state', 'generate_daily_arrivals', 'process_provider_queue',
    'process_er_queue', 'release_scheduled_patients_for_today',
    'release_returning_er_patients', 'is_weekday', 'next_weekday',
)])

## Extended Patient Generation

We override `generate_daily_arrivals` to produce our enriched `Patient` objects
(with demographics + clinical flags from `population.py`).
All of Sophia's queue functions still work because our Patient is a superset.

In [3]:
def generate_enriched_daily_arrivals(day: int, state: dict, next_patient_id: int) -> int:
    """
    Generate DAILY_PATIENTS enriched patients for one weekday.
    Replaces Sophia's generate_daily_arrivals with population.sample_patient.
    Queue logic (drop-in vs outpatient routing) matches Sophia's original.
    """
    for _ in range(cfg.DAILY_PATIENTS):
        patient_type = random.choices(
            list(cfg.PATIENT_TYPE_PROBS.keys()),
            weights=list(cfg.PATIENT_TYPE_PROBS.values())
        )[0]
        destination = random.choices(
            list(cfg.DESTINATION_PROBS.keys()),
            weights=list(cfg.DESTINATION_PROBS.values())
        )[0]

        # Draw enriched patient from population sampler
        patient = sample_patient(next_patient_id, day, destination, patient_type)
        next_patient_id += 1

        if destination == 'er':
            patient.critical_status = random.random() < cfg.ER_CRITICAL_PROB

        state['all_patients'].append(patient)
        state['patients_created'] += 1
        state['created_by_type'][patient_type] += 1
        state['created_by_destination'][destination] += 1

        if patient_type == 'outpatient':
            add_patient_to_today_queue(patient, state)
        elif destination == 'er':
            state['er_today'].append(patient)
        else:
            add_patient_to_today_queue(patient, state)

    return next_patient_id

print('generate_enriched_daily_arrivals defined')

generate_enriched_daily_arrivals defined


## Post-Provider Screening Step

Called immediately after a patient is *seen* by a provider.
Runs Steps 2–5 for all eligible cancer screenings.

In [ ]:
def run_post_provider_screening(patient: Patient, day: int, metrics: dict) -> None:
    """
    Execute screening steps for one patient who has just been seen by a provider.
    Updates patient object in place and records to metrics.

    Flow:
      1. Determine eligible screenings.
      2. If none / patient declines → handle_unscreened.
      3. For each eligible cancer → run_screening_step (lung includes pre-LDCT pathway).
      4. For each result → run cancer-specific follow-up.
    """
    metrics['n_patients'] += 1
    eligible = get_eligible_screenings(patient)

    if not eligible:
        metrics['n_unscreened'] += 1
        outcome = handle_unscreened(patient, day)
        if outcome == 'reschedule':
            metrics['n_reschedule'] += 1
            metrics['ltfu_unscreened'] += 1
        return

    metrics['n_eligible_any'] += 1

    for cancer in eligible:
        if not patient.active:
            break   # patient exited system mid-encounter (e.g. lung LTFU)

        # Pass metrics so lung pre-LDCT funnel counters are populated
        result = run_screening_step(patient, cancer, day, metrics=metrics)
        if result is None:
            continue

        record_screening(metrics, patient, cancer, result)

        # Route to cancer-specific follow-up
        if cancer == 'cervical':
            disposition = run_cervical_followup(patient, day, metrics)
        elif cancer == 'lung':
            disposition = run_lung_followup(patient, day, metrics)
        else:
            disposition = run_stub_followup(patient, cancer, result, day, metrics)

        if patient.exit_reason:
            record_exit(metrics, patient.exit_reason)

print('run_post_provider_screening defined')

## Extended Provider Queue Processor

Wraps Sophia's `process_provider_queue` to trigger screening after each *seen* event.

In [5]:
def process_provider_queue_with_screening(
    day: int, queue: deque, capacity: int,
    provider_name: str, state: dict, metrics: dict
) -> None:
    """
    Sophia's provider queue logic + screening step for every seen patient.
    """
    seen = 0

    while queue:
        patient = queue.popleft()

        if seen < capacity:
            seen += 1
            state['seen_by_destination'][provider_name] += 1
            log_day(state, day, f'Patient {patient.patient_id} seen by {provider_name}')

            # ── Screening step (Steps 2–5) ────────────────────────────────────
            run_post_provider_screening(patient, day, metrics)

        else:
            # Not seen today — reschedule (mirrors Sophia's logic)
            patient.wait_days += 1
            state['not_seen_by_destination'][provider_name] += 1

            if patient.patient_type == 'drop_in':
                patient.patient_type  = 'outpatient'
                patient.scheduled_day = next_weekday(day)
                state['future_schedule'][patient.scheduled_day].append(patient)
                state['converted_dropin_to_outpatient'] += 1
            else:
                patient.scheduled_day = next_weekday(day)
                state['future_schedule'][patient.scheduled_day].append(patient)

print('process_provider_queue_with_screening defined')

process_provider_queue_with_screening defined


## Main Daily Process

In [6]:
def daily_process_with_screening(env, state: dict, metrics: dict):
    """
    SimPy generator: drives one weekday at a time.
    Arrivals → queue processing (with screening) → advance clock.
    """
    import simpy
    next_patient_id = 1

    while env.now < cfg.SIM_DAYS:
        day = int(env.now)

        if not is_weekday(day):
            yield env.timeout(1)
            continue

        # 1. Release scheduled outpatients and returning ER patients
        release_scheduled_patients_for_today(day, state)
        release_returning_er_patients(day, state)

        # 2. Generate new enriched arrivals
        next_patient_id = generate_enriched_daily_arrivals(day, state, next_patient_id)

        # 3. Process provider queues (with integrated screening)
        process_provider_queue_with_screening(
            day, state['pcp_today'],        cfg.PROVIDER_CAPACITY['pcp'],
            'pcp',         state, metrics
        )
        process_provider_queue_with_screening(
            day, state['gyn_today'],        cfg.PROVIDER_CAPACITY['gynecologist'],
            'gynecologist', state, metrics
        )
        process_provider_queue_with_screening(
            day, state['specialist_today'], cfg.PROVIDER_CAPACITY['specialist'],
            'specialist',  state, metrics
        )

        # ER: use Sophia's ER processor (no screening in ER for now)
        process_er_queue(day, state)

        yield env.timeout(1)

print('daily_process_with_screening defined')

daily_process_with_screening defined


## Run Simulation

In [7]:
import simpy

def run_simulation(
    sim_days: int  = cfg.SIM_DAYS,
    seed:     int  = cfg.RANDOM_SEED,
):
    """
    Run the full end-to-end simulation.

    Returns
    -------
    state   : Sophia's arrivals state dict
    metrics : Steps 2–6 metrics dict
    """
    random.seed(seed)

    env     = simpy.Environment()
    state   = initialize_state()
    metrics = initialize_metrics()

    env.process(daily_process_with_screening(env, state, metrics))
    env.run(until=sim_days)

    return state, metrics

print('run_simulation defined')

run_simulation defined


## Quick Test Run — 1 Year, 1 Replication

In [8]:
print('Running 1-year simulation...')
state, metrics = run_simulation(sim_days=365, seed=42)
print(f"Done. Patients created: {state['patients_created']:,}")
print()
print_summary(metrics)

Running 1-year simulation...
Done. Patients created: 52,200

ARRIVAL / ACCESS SUMMARY


KeyError: 'patients_created'

## Arrivals Summary (Sophia's metrics)

In [9]:
# Re-use Sophia's print_summary for the arrivals layer
print_summary(state)   # this calls Sophia's function (loaded via %run)

ARRIVAL / ACCESS SUMMARY
Total patients created:               52200

Created by type:
  outpatient:                         36737
  drop_in:                            15463

Created by destination:
  pcp:                                18133
  gynecologist:                       13195
  specialist:                         10433
  er:                                 10439

Seen by destination:
  pcp:                                10440
  gynecologist:                       7830
  specialist:                         5220
  er:                                 6525

Drop-ins converted to outpatients:    12356
Critical ER returned next day:        413348
Noncritical ER scheduled outpatient:  98495

Outpatient showups:                   2473858
Outpatient no-shows:                  0


## Trace a Few Patients

In [10]:
from metrics import print_patient_trace

# Show first 3 patients who had any screening event
screened_patients = [p for p in state['all_patients'] if p.event_log]
print_patient_trace(screened_patients, n=3)


── Patient 1 | age=33 | destination=pcp ──
  Day     0: SCREEN cervical via hpv_alone
  Day     0: RESULT cervical: NORMAL
  Day     0: ROUTE cervical NORMAL → routine surveillance

── Patient 2 | age=58 | destination=gynecologist ──
  Day     0: SCREEN cervical via hpv_alone
  Day     0: RESULT cervical: NORMAL
  Day     0: ROUTE cervical NORMAL → routine surveillance
  Day     0: SCREEN breast via mammogram
  Day     0: RESULT breast: NEGATIVE
  Day     0: FOLLOWUP breast NEGATIVE → routine surveillance
  Day     0: SCREEN colorectal via colonoscopy
  Day     0: RESULT colorectal: NEGATIVE
  Day     0: FOLLOWUP colorectal NEGATIVE → routine surveillance

── Patient 3 | age=51 | destination=pcp ──
  Day     0: SCREEN cervical via co_test
  Day     0: RESULT cervical: NORMAL
  Day     0: ROUTE cervical NORMAL → routine surveillance
  Day     0: SCREEN breast via mammogram
  Day     0: RESULT breast: NEGATIVE
  Day     0: FOLLOWUP breast NEGATIVE → routine surveillance
  Day     0: SCR